In [1]:
# !pip install earthengine-api
# !pip install geemap
# !pip install sidecar

In [1]:
import ee
import pandas as pd

In [2]:
# sc = Sidecar(title='Sidecar Output')

In [3]:
ee.Authenticate()

True

In [4]:
# Initialize the Earth Engine API
ee.Initialize()

In [54]:
df = pd.read_csv("example_points.csv", index_col=0)
df

,longitude,latitude,date
2,-79.962909,36.618243,2024-12-06
3,-79.962976,36.618235,2024-12-06
4,-79.963042,36.618227,2024-12-06
12,-79.962852,36.618305,2024-12-06
13,-79.962918,36.618297,2024-12-06
...,...,...,...
152,-79.963501,36.618888,2024-12-06
161,-79.963377,36.618957,2024-12-06
162,-79.963444,36.618949,2024-12-06
163,-79.963511,36.618941,2024-12-06


In [55]:
def create_feature_collection(df):
    """Convert the CSV into an Earth Engine FeatureCollection"""
    features = []
    for index, row in df.iterrows():
        lat = row["latitude"]
        lon = row["longitude"]
        date = row["date"]  # Assuming date format is 'YYYY-MM-DD'

        # Create an ee.Feature for each row
        feature = ee.Feature(ee.Geometry.Point([lon, lat]), {"date": date})
        features.append(feature)

    # Return an Earth Engine FeatureCollection
    return ee.FeatureCollection(features)

In [56]:
# Create FeatureCollection of all points from CSV
points_fc = create_feature_collection(df)

In [58]:
# Map = geemap.Map(center=[40,-100], zoom=4)
# Map.addLayer(points_fc, {}, "Points")
# Map

# # with sc:
# #     display(Map)

In [119]:
def prep_features(feature):
    date = ee.Date(feature.get("date"))

    # Define the time range: 30 days before the given date
    start_date = date.advance(-30, "day")
    end_date = date

    # Create a 2,000 meter buffer around the point (circular buffer)
    point = feature.geometry()
    buffer = point.buffer(2000)  # Buffer is in meters (2 km radius)

    s2_collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    cloud_score_collection = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

    latest_image = (
        s2_collection.filterBounds(buffer)
        .filterDate(start_date, end_date)
        .filter(
            "CLOUDY_PIXEL_PERCENTAGE < 15"
        )  # exclude images where more than 5% of pixels in image are clouds
        .sort("system:time_start", False)  # sort so we can take most recent image
        .linkCollection(cloud_score_collection, ["cs_cdf"])  # bring in cloud score
    ).first()

    feature = feature.set(
        {
            "image_id": latest_image.get("system:id"),
            "image_date": ee.Date(latest_image.get("system:time_start")).format(
                "YYYY-MM-dd HH:mm:ss"
            ),
        }
    )

    masked_image = latest_image.updateMask(
        latest_image.select("cs_cdf").gte(0.6)
    ).updateMask(  # mask out pixels where cloud score is too high
        latest_image.select("SCL").eq(6)
    )  # filter to water area

    # add NDVI bands
    for band in ["B4", "B5", "B6", "B7"]:
        masked_image = masked_image.addBands(
            masked_image.normalizedDifference(["B8", band]).rename(f"NDVI_{band}")
        )

    # add ratio bands
    green_red_ratio = (
        masked_image.select("B3").divide(masked_image.select("B4")).rename("green_red_ratio")
    )
    green_blue_ratio = (
        masked_image.select("B3").divide(masked_image.select("B2")).rename("green_blue_ratio")
    )
    red_blue_ratio = (
        masked_image.select("B4").divide(masked_image.select("B2")).rename("red_blue_ratio")
    )
    masked_image = masked_image.addBands([green_red_ratio, green_blue_ratio, red_blue_ratio])

    # band means
    means = masked_image.select("B.*", "WVP", "AOT", "NDVI.*", ".*_ratio").reduceRegion(
        reducer=ee.Reducer.mean(), geometry=buffer, scale=10, maxPixels=1e8
    )

    # green percentiles
    percentiles = masked_image.select("B3").reduceRegion(
        reducer=ee.Reducer.percentile([5, 95]), geometry=buffer, scale=10, maxPixels=1e8
    )

    # percent water (use unmasked image)
    mean_water = (
        latest_image.select("SCL")
        .eq(6)
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=buffer, scale=10, maxPixels=1e8)
        .rename(["SCL"], ["percent_water"])
    )

    # inputs for AOT range
    aot_minmax = masked_image.select("AOT").reduceRegion(
        reducer=ee.Reducer.minMax(),  # Get both min and max values in one step
        geometry=buffer,  # Area of interest (buffer)
        scale=10,  # Scale (10m for Sentinel-2)
        maxPixels=1e8,
    )

    stats = means.combine(percentiles).combine(mean_water).combine(aot_minmax)
    return feature.set(stats)

In [120]:
points_fc = create_feature_collection(df)
results_fc = points_fc.map(prep_features)

In [121]:
# results_fc.getInfo()

In [122]:
feature_results = [r["properties"] for r in results_fc.getInfo()["features"]]
feature_df = pd.DataFrame(feature_results)
feature_df

,AOT,AOT_max,AOT_min,B1,B11,B12,B2,B3,B3_p5,B3_p95,...,NDVI_B6,NDVI_B7,WVP,date,green_blue_ratio,green_red_ratio,image_date,image_id,percent_water,red_blue_ratio
0,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
1,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
2,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
3,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
4,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
113,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
114,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685
115,74.2,75,74,128.8,328.466667,230.2,133.916667,125.4,63,215.5,...,0.174822,0.25704,242,2024-12-06,0.930155,1.032048,2024-12-03 16:13:15,COPERNICUS/S2_SR_HARMONIZED/20241203T160539_20...,0.000483,0.933685


In [36]:
# Map = geemap.Map(center=[37.56, -101.34], zoom=9)
# Map.addLayer(masked_image.select(['B4', 'B3', 'B2']), {'min': 0, 'max': 3000})
# Map